# Additional experiments

Numerical diagnostics and explanatory controls: mesh sensitivity, local Gaussian and row-sum mechanisms, fixed-sample convergence errors, and local minima versus interval endpoints.

Run the cells in order; all required functions are imported below. Every displayed figure is saved automatically under `outputs/pdfs/06_additional_experiments/`.

Source: [KDE chapter](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex).


In [ ]:
from _setup import setup_notebook
import hashlib
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from IPython.display import display
from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta
from kde_tools.energy_minimization import EnergySearchConfig, energy_search_settings
from kde_tools.experiment_inputs import REFERENCE_SAMPLE_PATH, load_reference_sample
from kde_tools.fourier.similarities import (
    DEFAULT_FREQUENCY_CUTOFF,
    DEFAULT_FREQUENCY_INTERVALS,
    beta2_divergence_matrix_fourier,
)
from kde_tools.joint_experiments import (
    inverse_psi_beta,
    local_custom_beta_metric,
    psi_beta,
    run_custom_beta_paths,
    run_joint_limit_paths,
)
from kde_tools.perron import perron_peak_index, perron_vector_from_divergence
from kde_tools.qhat import GaussianQhatObjective, minimize_qhat

paths = setup_notebook("06_additional_experiments")
code_directory = paths.code
pdf_output_directory = paths.pdfs
ENERGY_SEARCH = EnergySearchConfig()
SEED = 20260823
N_DATA = 200
H_MIN, H_MAX = (0.01, 2.0)
BETA = 2.0
N_VALUES = np.arange(100, 3001, 100)
FREQUENCY_CUTOFF = DEFAULT_FREQUENCY_CUTOFF
FREQUENCY_INTERVALS = DEFAULT_FREQUENCY_INTERVALS
FFT_HALF_WIDTH = np.pi * FREQUENCY_INTERVALS / FREQUENCY_CUTOFF
FFT_SPATIAL_POINTS = 2 * FREQUENCY_INTERVALS
plt.style.use("seaborn-v0_8-whitegrid")


## Reference calculations for the diagnostics

These calls construct the fixed-sample reference results used below. This notebook is independent of notebook 03 and uses the same shared Python functions.


In [ ]:
samples = np.random.default_rng(SEED).normal(size=N_DATA)
squared_differences = (samples[:, None] - samples[None, :]) ** 2
h_infinity = minimize_qhat(samples, H_MIN, H_MAX, r=0.0, config=ENERGY_SEARCH)["global_minimum"][
    "h"
]
LAMBDA_RULES = {
    "$N^{1/2}$ (admissible)": lambda N: N**0.5,
    "$N$ (admissible)": lambda N: float(N),
    "$N^{3/2}$ (admissible)": lambda N: N**1.5,
    "$1$ (not localized)": lambda N: 1.0,
    "$N^2$ (mesh-scale)": lambda N: float(N**2),
    "$N^3$ (collapse)": lambda N: float(N**3),
}
REGIME_ORDER = list(LAMBDA_RULES)
REGIME_COLORS = dict(
    zip(REGIME_ORDER, ["#0072B2", "#009E73", "#CC79A7", "#E69F00", "#7A7A7A", "#D55E00"])
)
results, largest_grid_vectors, elapsed = run_joint_limit_paths(
    beta=BETA,
    half_width=FFT_HALF_WIDTH,
    spatial_points=FFT_SPATIAL_POINTS,
    h_max=H_MAX,
    h_min=H_MIN,
    lambda_rules=LAMBDA_RULES,
    grid_sizes=N_VALUES,
    regime_order=REGIME_ORDER,
    h_reference=h_infinity,
    samples=samples,
)
plot_ticks = np.array([100, 500, 1000, 1500, 2000, 2500, 3000])


## Local Gaussian approximation and mesh resolution


In [ ]:
CUSTOM_BETA = 1 / 7
if not np.isfinite(CUSTOM_BETA) or CUSTOM_BETA <= 0.0:
    raise ValueError("CUSTOM_BETA must be positive and finite.")
CUSTOM_BETA_N_VALUES = N_VALUES.copy()
CUSTOM_ALPHA_VALUES = np.round(np.arange(1.0, 3.01, 0.5), 1)
CUSTOM_BETA_LAMBDA_RULES = {"$N^{0.5}$": lambda N: N**0.5}
for alpha in CUSTOM_ALPHA_VALUES:
    if np.isclose(alpha, 2.0):
        label = "$N^{2.0}$ (mesh-scale)"
    elif alpha > 2.0:
        label = f"$N^{{{alpha:.1f}}}$ (collapse)"
    else:
        label = f"$N^{{{alpha:.1f}}}$"
    CUSTOM_BETA_LAMBDA_RULES[label] = lambda N, alpha=alpha: N**alpha
CUSTOM_BETA_LAMBDA_RULES["$1$ (not localized)"] = lambda N: 1.0
CUSTOM_BETA_REGIME_ORDER = list(CUSTOM_BETA_LAMBDA_RULES)
cmap = plt.colormaps["turbo"]
CUSTOM_BETA_REGIME_COLORS = {
    regime: cmap(i / (len(CUSTOM_BETA_REGIME_ORDER) - 1))
    for i, regime in enumerate(CUSTOM_BETA_REGIME_ORDER)
}
custom_beta_continuous_minimum = minimize_qhat(
    samples, H_MIN, H_MAX, r=1 - CUSTOM_BETA / 2, config=ENERGY_SEARCH
)
if custom_beta_continuous_minimum["diagnostics"]["resolution_stable"] is False:
    raise ArithmeticError("Energy search did not stabilize; increase ENERGY_SEARCH resolution.")
custom_beta_reference_candidates = np.array(
    [H_MIN, custom_beta_continuous_minimum["global_minimum"]["h"], H_MAX]
)
custom_beta_reference_values = np.array(
    [
        local_custom_beta_metric(h, squared_differences, beta=CUSTOM_BETA)
        for h in custom_beta_reference_candidates
    ]
)
custom_beta_h_infinity = float(
    custom_beta_reference_candidates[np.argmin(custom_beta_reference_values)]
)
diagnostic_rows = []
diagnostic_start_time = time.perf_counter()
psi_min = psi_beta(H_MIN, CUSTOM_BETA)
psi_max = psi_beta(H_MAX, CUSTOM_BETA)
for current_N in CUSTOM_BETA_N_VALUES:
    current_N = int(current_N)
    current_h_grid = bandwidth_grid_linear_in_psi_beta(
        h_min=H_MIN, h_max=H_MAX, beta=CUSTOM_BETA, grid_size=current_N
    )
    delta_psi = (psi_max - psi_min) / (current_N - 1)
    current_divergence = beta2_divergence_matrix_fourier(
        samples, current_h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
    )
    reference_index = int(np.argmin(np.abs(current_h_grid - custom_beta_h_infinity)))
    reference_h = float(current_h_grid[reference_index])
    Q_i = local_custom_beta_metric(reference_h, squared_differences, beta=CUSTOM_BETA)
    if not np.isfinite(Q_i) or Q_i <= 0.0:
        raise RuntimeError(f"Invalid local metric Q_i={Q_i} at N={current_N}.")
    grid_offsets = (np.arange(current_N) - reference_index) * delta_psi
    divergence_row = np.maximum(current_divergence[reference_index, :], 0.0)
    for regime, lambda_rule in CUSTOM_BETA_LAMBDA_RULES.items():
        lambda_value = float(lambda_rule(current_N))
        similarity_row = np.exp(-lambda_value * divergence_row)
        actual_row_sum = float(np.sum(similarity_row))
        diagonal_mass = float(similarity_row[reference_index])
        off_diagonal_mass = float(actual_row_sum - diagonal_mass)
        gaussian_row = np.exp(-lambda_value * Q_i * grid_offsets**2)
        gaussian_row_sum = float(np.sum(gaussian_row))
        continuous_integral = float(np.sqrt(np.pi / (lambda_value * Q_i)))
        actual_integral_ratio = float(delta_psi * actual_row_sum / continuous_integral)
        gaussian_quadrature_ratio = float(delta_psi * gaussian_row_sum / continuous_integral)
        local_approximation_ratio = float(actual_row_sum / gaussian_row_sum)
        mesh_to_width = float(np.sqrt(lambda_value * Q_i) * delta_psi)
        effective_grid_points = float(1.0 / mesh_to_width)
        nearest_values = []
        if reference_index > 0:
            nearest_values.append(similarity_row[reference_index - 1])
        if reference_index < current_N - 1:
            nearest_values.append(similarity_row[reference_index + 1])
        nearest_neighbor_similarity = float(np.mean(nearest_values))
        diagnostic_rows.append(
            {
                "beta": CUSTOM_BETA,
                "N": current_N,
                "regime": regime,
                "lambda": lambda_value,
                "reference_index": reference_index,
                "reference_h": reference_h,
                "Q_i": Q_i,
                "delta_psi": delta_psi,
                "mesh_to_width": mesh_to_width,
                "effective_grid_points": effective_grid_points,
                "actual_row_sum": actual_row_sum,
                "gaussian_row_sum": gaussian_row_sum,
                "continuous_integral": continuous_integral,
                "actual_integral_ratio": actual_integral_ratio,
                "gaussian_quadrature_ratio": gaussian_quadrature_ratio,
                "local_approximation_ratio": local_approximation_ratio,
                "off_diagonal_mass": off_diagonal_mass,
                "nearest_neighbor_similarity": nearest_neighbor_similarity,
            }
        )
diagnostic_results = pd.DataFrame(diagnostic_rows)
diagnostic_results["regime"] = pd.Categorical(
    diagnostic_results["regime"], categories=CUSTOM_BETA_REGIME_ORDER, ordered=True
)
diagnostic_results = diagnostic_results.sort_values(["regime", "N"]).reset_index(drop=True)
diagnostic_elapsed = time.perf_counter() - diagnostic_start_time
print(f"Completed diagnostic sweep for beta={CUSTOM_BETA:g} in {diagnostic_elapsed:.2f} seconds.")
print(f"Continuous reference: h_infinity = {custom_beta_h_infinity:.8f}")
display(
    diagnostic_results.pivot(index="N", columns="regime", values="gaussian_quadrature_ratio").round(
        6
    )
)
fig, ax = plt.subplots(figsize=(10, 6))
for regime in CUSTOM_BETA_REGIME_ORDER:
    subset = diagnostic_results[diagnostic_results["regime"] == regime]
    ax.plot(
        subset["N"],
        subset["gaussian_quadrature_ratio"],
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=regime,
        color=CUSTOM_BETA_REGIME_COLORS[regime],
    )
ax.axhline(1.0, linestyle="--", linewidth=1.5, color="black", label="continuum integral")
ax.set_xscale("log")
ax.set_xlabel("number of bandwidths $N$")
ax.set_ylabel("$\\Delta_\\psi \\sum_j g_i(\\delta_{ij})\\,/\\,\\sqrt{\\pi/(\\lambda Q_i)}$")
ax.set_title(f"Failure of the continuum quadrature regime, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(10, 6))
for regime in CUSTOM_BETA_REGIME_ORDER:
    subset = diagnostic_results[diagnostic_results["regime"] == regime]
    ax.plot(
        subset["N"],
        subset["mesh_to_width"],
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=regime,
        color=CUSTOM_BETA_REGIME_COLORS[regime],
    )
ax.axhline(1.0, linestyle="--", linewidth=1.5, color="black")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("number of bandwidths $N$")
ax.set_ylabel("$\\sqrt{\\lambda Q_i}\\,\\Delta_\\psi$")
ax.set_title(f"Mesh scale versus localization scale, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(10, 6))
for regime in CUSTOM_BETA_REGIME_ORDER:
    subset = diagnostic_results[diagnostic_results["regime"] == regime]
    values = np.maximum(subset["off_diagonal_mass"].to_numpy(), 1e-300)
    ax.plot(
        subset["N"],
        values,
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=regime,
        color=CUSTOM_BETA_REGIME_COLORS[regime],
    )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("number of bandwidths $N$")
ax.set_ylabel("$\\sum_{j\\neq i} S_{ij}$")
ax.set_title(f"Off-diagonal similarity mass, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(10, 6))
for regime in CUSTOM_BETA_REGIME_ORDER:
    subset = diagnostic_results[diagnostic_results["regime"] == regime]
    values = np.maximum(subset["nearest_neighbor_similarity"].to_numpy(), 1e-300)
    ax.plot(
        subset["N"],
        values,
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=regime,
        color=CUSTOM_BETA_REGIME_COLORS[regime],
    )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("number of bandwidths $N$")
ax.set_ylabel("mean nearest-neighbor similarity")
ax.set_title(f"Nearest-neighbor collapse, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
criterion_results = diagnostic_results.copy()
criterion_results["alpha"] = np.log(criterion_results["lambda"]) / np.log(criterion_results["N"])
criterion_results["alpha"] = criterion_results["alpha"].round(10)
criterion_results = criterion_results[
    (criterion_results["alpha"] >= 1.0) & (criterion_results["alpha"] <= 3.0)
].copy()
eps = 1e-300
criterion_results["local_ratio_safe"] = np.maximum(
    criterion_results["local_approximation_ratio"], eps
)
criterion_results["mesh_ratio_safe"] = np.maximum(
    criterion_results["gaussian_quadrature_ratio"], eps
)
criterion_results["E_local"] = np.abs(np.log(criterion_results["local_ratio_safe"]))
criterion_results["E_mesh"] = np.abs(np.log(criterion_results["mesh_ratio_safe"]))
criterion_results["C"] = np.sqrt(
    criterion_results["E_local"] ** 2 + criterion_results["E_mesh"] ** 2
)
fig, ax = plt.subplots(figsize=(11, 7))
for current_N in sorted(criterion_results["N"].unique()):
    subset = criterion_results[criterion_results["N"] == current_N].sort_values("alpha")
    ax.plot(
        subset["alpha"],
        subset["C"],
        marker="o",
        markersize=4,
        linewidth=1.5,
        label=f"$N={current_N}$",
    )
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel("$C_N(\\alpha)$")
ax.set_title(f"Localization--mesh criterion, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
largest_N = int(criterion_results["N"].max())
largest_N_results = (
    criterion_results[criterion_results["N"] == largest_N].sort_values("alpha").copy()
)
optimal_row = largest_N_results.loc[largest_N_results["C"].idxmin()]
optimal_alpha = float(optimal_row["alpha"])
optimal_C = float(optimal_row["C"])
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(largest_N_results["alpha"], largest_N_results["C"], marker="o", markersize=5, linewidth=2)
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.axvline(
    optimal_alpha,
    linestyle=":",
    linewidth=1.8,
    color="black",
    label=f"$\\alpha_{{N}}^\\star={optimal_alpha:.1f}$",
)
ax.scatter([optimal_alpha], [optimal_C], s=70, zorder=5)
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel(f"$C_{{{largest_N}}}(\\alpha)$")
ax.set_title(f"Criterion at $N={largest_N}$, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
print(
    f"For N={largest_N}, the minimizing exponent is alpha* = {optimal_alpha:.1f}, with C = {optimal_C:.8g}."
)
optimal_alpha_by_N = (
    criterion_results.loc[
        criterion_results.groupby("N")["C"].idxmin(), ["N", "alpha", "C", "E_local", "E_mesh"]
    ]
    .sort_values("N")
    .reset_index(drop=True)
)
optimal_alpha_by_N = optimal_alpha_by_N.rename(columns={"alpha": "optimal_alpha", "C": "minimum_C"})
display(optimal_alpha_by_N.round(8))


### Zoom on the positive-power regimes

Here we retain all positive-power exponents $\alpha\in\{1/2,1,3/2,2,3\}$, restrict attention to $N\geq1000$, and zoom in around the continuous reference $h_\infty$. Solid lines denote the three admissible regimes $\alpha<2$; dashed lines denote the mesh-scale case $\alpha=2$ and the collapsing case $\alpha=3$. Their visual proximity to $h_\infty$ should therefore not be interpreted as convergence to the same continuous localized-kernel limit. The small oscillations are caused by projecting the target onto a finite bandwidth grid.


In [ ]:
FOCUSED_REGIMES = [
    REGIME_ORDER[0],
    REGIME_ORDER[1],
    REGIME_ORDER[2],
    REGIME_ORDER[4],
    REGIME_ORDER[5],
]
ZOOM_N_MIN = 1000
zoom_results = results[
    results["regime"].isin(FOCUSED_REGIMES) & (results["N"] >= ZOOM_N_MIN)
].copy()
REGIME_MARKERS = dict(zip(FOCUSED_REGIMES, ["o", "s", "^", "D", "X"]))
fig, ax = plt.subplots(figsize=(9.5, 5.8))
for regime in FOCUSED_REGIMES:
    subset = zoom_results[zoom_results["regime"] == regime]
    admissible = "admissible" in regime
    ax.plot(
        subset["N"],
        subset["selected_h"],
        marker=REGIME_MARKERS[regime],
        markersize=5.5,
        linewidth=2.2 if admissible else 1.7,
        linestyle="-" if admissible else "--",
        markerfacecolor=REGIME_COLORS[regime] if admissible else "white",
        markeredgewidth=1.4,
        color=REGIME_COLORS[regime],
        label=regime,
    )
zoom_min = min(h_infinity, zoom_results["selected_h"].min())
zoom_max = max(h_infinity, zoom_results["selected_h"].max())
zoom_padding = 0.06 * (zoom_max - zoom_min)
ax.axhline(
    h_infinity,
    color="black",
    linestyle=":",
    linewidth=2.0,
    label=f"continuous $h_\\infty={h_infinity:.4f}$",
)
ax.set_ylim(zoom_min - zoom_padding, zoom_max + zoom_padding)
ax.set(
    xlabel="number of bandwidths $N$",
    ylabel="selected bandwidth $h_{\\mathrm{spec},N}$",
    title="Positive-power regimes near the continuous reference",
)
ax.set_xticks(plot_ticks[plot_ticks >= ZOOM_N_MIN], labels=plot_ticks[plot_ticks >= ZOOM_N_MIN])
ax.legend(ncol=2, frameon=True)
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.8))
for regime in REGIME_ORDER:
    subset = results[results["regime"] == regime]
    ax.plot(
        subset["N"],
        subset["neighbor_exponent"],
        marker="o",
        linewidth=2.0,
        color=REGIME_COLORS[regime],
        label=regime,
    )
ax.axhline(1.0, color="black", linestyle=":", linewidth=1.5)
ax.set(
    yscale="log",
    xlabel="number of bandwidths $N$",
    ylabel="$\\lambda_ND_{i,i+1}$ near $h_\\infty$",
    title="Mesh-resolution diagnostic (must tend to 0)",
)
ax.set_xticks(plot_ticks, labels=plot_ticks)
ax.legend(ncol=2, frameon=True)
fig.tight_layout()
plt.show()


In [ ]:
final_N = int(N_VALUES[-1])
final_diagnostics = results[results["N"] == final_N][
    [
        "regime",
        "lambda",
        "lambda_over_N2",
        "selected_h",
        "absolute_error",
        "global_exponent",
        "neighbor_exponent",
        "neighbor_similarity",
    ]
].copy()
admissible_final = final_diagnostics[final_diagnostics["regime"].isin(REGIME_ORDER[:3])]
admissible_spread = admissible_final["selected_h"].max() - admissible_final["selected_h"].min()
display(final_diagnostics.round(7))
print(f"At N={final_N}, the admissible selectors span {admissible_spread:.6f} in bandwidth.")
fig, ax = plt.subplots(figsize=(10.5, 5.2))
for regime in REGIME_ORDER:
    h_grid, vector = largest_grid_vectors[regime]
    admissible = "admissible" in regime
    ax.plot(
        h_grid,
        vector / vector.max(),
        color=REGIME_COLORS[regime],
        linewidth=2.0 if admissible else 1.5,
        linestyle="-" if admissible else "--",
        label=regime,
    )
ax.axvline(h_infinity, color="black", linestyle=":", linewidth=2.0)
ax.set(
    xscale="log",
    xlabel="bandwidth $h$",
    ylabel="Perron component / maximum",
    title=f"Perron profiles at the finest grid ($N={final_N}$)",
)
ax.legend(ncol=2)
plt.show()


## Mesh-robustness stress test for the six regimes

We use the same six exponents as above:

$$\alpha\in\{0,1/2,1,3/2,2,3\},\qquad \lambda_N=N^\alpha.$$

For each exponent, one point in an otherwise uniform $\psi_2=\log h$ grid is moved toward its left neighbor near $h=0.7$, far from the continuous prediction $h_\infty$. The endpoints, number of candidates, sample, bandwidth family, and all other grid points remain fixed.

The retained-gap fraction runs from one (the original uniform grid) down to $0.1$. A jump of the selected bandwidth toward $0.7$ indicates that the Perron vector has locked onto the artificial close pair.

**Scope.** This is a robustness stress test, not a counterexample under the theorem's exact uniform-grid assumptions.


In [ ]:
MESH_STRESS_N = 400
MESH_TARGET_H = 0.7
MESH_ALPHA_VALUES = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0])
MESH_ALPHA_COLORS = plt.colormaps["tab20"](np.arange(len(MESH_ALPHA_VALUES)))
MESH_GAP_FRACTIONS = np.array([1.0, 0.9, 0.75, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1])
mesh_base_grid = bandwidth_grid_linear_in_psi_beta(H_MIN, H_MAX, beta=BETA, grid_size=MESH_STRESS_N)
mesh_base_psi = np.log(mesh_base_grid)
mesh_delta_psi = mesh_base_psi[1] - mesh_base_psi[0]
mesh_close_pair_left = int(np.argmin(np.abs(mesh_base_grid - MESH_TARGET_H)))
mesh_close_pair_left = min(mesh_close_pair_left, MESH_STRESS_N - 2)
mesh_rows = []
mesh_alpha3_profiles = {}
for gap_fraction in MESH_GAP_FRACTIONS:
    perturbed_psi = mesh_base_psi.copy()
    perturbed_psi[mesh_close_pair_left + 1] = (
        perturbed_psi[mesh_close_pair_left] + gap_fraction * mesh_delta_psi
    )
    h_grid = np.exp(perturbed_psi)
    divergence = beta2_divergence_matrix_fourier(
        samples, h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
    )
    for alpha in MESH_ALPHA_VALUES:
        alpha = float(alpha)
        lambda_value = float(MESH_STRESS_N**alpha)
        vector = perron_vector_from_divergence(divergence, lambda_value)
        selected_index = int(perron_peak_index(vector))
        mesh_rows.append(
            {
                "gap_fraction": gap_fraction,
                "alpha": alpha,
                "lambda": lambda_value,
                "selected_h": float(h_grid[selected_index]),
                "selected_index": selected_index,
                "close_pair_h_left": float(h_grid[mesh_close_pair_left]),
                "close_pair_h_right": float(h_grid[mesh_close_pair_left + 1]),
                "close_pair_similarity": float(
                    np.exp(
                        -lambda_value * divergence[mesh_close_pair_left, mesh_close_pair_left + 1]
                    )
                ),
            }
        )
        if np.isclose(alpha, 3.0) and gap_fraction in {1.0, 0.2}:
            mesh_alpha3_profiles[gap_fraction] = (h_grid.copy(), vector.copy())
mesh_stress_results = pd.DataFrame(mesh_rows)
mesh_selected_table = mesh_stress_results.pivot(
    index="gap_fraction", columns="alpha", values="selected_h"
)
print(f"Completed {len(mesh_stress_results)} mesh-stress selector computations.")
display(mesh_selected_table.loc[[1.0, 0.6, 0.4, 0.3, 0.2, 0.1]].round(6))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.4))
for alpha_index, alpha in enumerate(MESH_ALPHA_VALUES):
    subset = mesh_stress_results[mesh_stress_results["alpha"] == alpha].sort_values(
        "gap_fraction", ascending=False
    )
    axes[0].plot(
        subset["gap_fraction"],
        subset["selected_h"],
        color=MESH_ALPHA_COLORS[alpha_index],
        linewidth=1.8,
        marker="o",
        markersize=4.8,
        markeredgecolor="white",
        markeredgewidth=0.35,
        label=f"$\\alpha={alpha:g}$",
    )
axes[0].axhline(
    h_infinity, color="black", linestyle=":", linewidth=2.0, label=f"$h_\\infty={h_infinity:.4f}$"
)
axes[0].axhline(
    MESH_TARGET_H,
    color="#D55E00",
    linestyle=":",
    linewidth=1.5,
    label=f"perturbed region $h\\approx{MESH_TARGET_H}$",
)
axes[0].invert_xaxis()
axes[0].set(
    xlabel="retained fraction of the standard local grid gap",
    ylabel="selected bandwidth $h_{\\mathrm{spec}}$",
    title="Mesh sensitivity for $\\alpha\\in\\{0,1/2,1,3/2,2,3\\}$",
)
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=4, fontsize=8.5, frameon=True)
alpha3_color = MESH_ALPHA_COLORS[-1]
for gap_fraction, linestyle in [(1.0, "-"), (0.2, "--")]:
    h_grid, vector = mesh_alpha3_profiles[gap_fraction]
    axes[1].plot(
        h_grid,
        vector / vector.max(),
        color=alpha3_color,
        linestyle=linestyle,
        linewidth=2.6,
        label=(
            "$\\alpha=3$, uniform grid" if gap_fraction == 1.0 else "$\\alpha=3$, local gap = 20%"
        ),
    )
axes[1].axvline(
    h_infinity, color="black", linestyle=":", linewidth=2.0, label=f"$h_\\infty={h_infinity:.4f}$"
)
axes[1].axvline(
    MESH_TARGET_H,
    color="#D55E00",
    linestyle=":",
    linewidth=1.5,
    label=f"perturbed region $h\\approx{MESH_TARGET_H}$",
)
axes[1].set(
    xscale="log",
    xlabel="bandwidth $h$",
    ylabel="Perron component / maximum",
    title="$\\alpha=3$ locks onto the artificial close pair",
)
axes[1].legend()
fig.suptitle(
    f"Mesh-robustness stress test on one fixed sample ($N={MESH_STRESS_N}$, $n={N_DATA}$, one perturbed grid point)",
    fontsize=14,
)
fig.subplots_adjust(bottom=0.31, wspace=0.22)
plt.show()


## Mesh stress across grid sizes and exponents


In [ ]:
MESH_TARGET_H = 0.7
MESH_STRESS_N_VALUES = np.array([200, 400, 800, 1200])
MESH_ALPHA_VALUES = np.concatenate([np.array([0.0, 0.5]), np.round(np.arange(1.0, 3.01, 0.1), 1)])
MESH_GAP_FRACTIONS = np.array([1.0, 0.9, 0.75, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1])
mesh_cmap = plt.colormaps["turbo"]
MESH_ALPHA_COLORS = {
    float(alpha): mesh_cmap(i / (len(MESH_ALPHA_VALUES) - 1))
    for i, alpha in enumerate(MESH_ALPHA_VALUES)
}
mesh_rows = []
mesh_start_time = time.perf_counter()
for mesh_N in MESH_STRESS_N_VALUES:
    mesh_N = int(mesh_N)
    mesh_base_grid = bandwidth_grid_linear_in_psi_beta(
        H_MIN, H_MAX, beta=CUSTOM_BETA, grid_size=mesh_N
    )
    mesh_base_psi = np.array([psi_beta(h, CUSTOM_BETA) for h in mesh_base_grid])
    mesh_delta_psi = float(mesh_base_psi[1] - mesh_base_psi[0])
    mesh_close_pair_left = int(np.argmin(np.abs(mesh_base_grid - MESH_TARGET_H)))
    mesh_close_pair_left = max(1, mesh_close_pair_left)
    mesh_close_pair_left = min(mesh_close_pair_left, mesh_N - 3)
    for gap_fraction in MESH_GAP_FRACTIONS:
        gap_fraction = float(gap_fraction)
        perturbed_psi = mesh_base_psi.copy()
        perturbed_psi[mesh_close_pair_left + 1] = (
            perturbed_psi[mesh_close_pair_left] + gap_fraction * mesh_delta_psi
        )
        h_grid = np.array([inverse_psi_beta(psi, CUSTOM_BETA) for psi in perturbed_psi])
        divergence = beta2_divergence_matrix_fourier(
            samples, h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
        )
        i = mesh_close_pair_left
        close_pair_divergence = float(max(divergence[i, i + 1], 0.0))
        regular_pair_divergence = float(max(divergence[i - 1, i], 0.0))
        Q_i = local_custom_beta_metric(float(h_grid[i]), squared_differences, beta=CUSTOM_BETA)
        for alpha in MESH_ALPHA_VALUES:
            alpha = float(alpha)
            lambda_value = float(mesh_N**alpha)
            vector = perron_vector_from_divergence(divergence, lambda_value)
            selected_index = int(perron_peak_index(vector))
            selected_h = float(h_grid[selected_index])
            close_pair_similarity = float(np.exp(-lambda_value * close_pair_divergence))
            regular_pair_similarity = float(np.exp(-lambda_value * regular_pair_divergence))
            log_similarity_ratio = float(
                lambda_value * (regular_pair_divergence - close_pair_divergence)
            )
            predicted_log_similarity_ratio = float(
                lambda_value * Q_i * mesh_delta_psi**2 * (1.0 - gap_fraction**2)
            )
            lambda_delta_psi_squared = float(lambda_value * mesh_delta_psi**2)
            mesh_rows.append(
                {
                    "N": mesh_N,
                    "gap_fraction": gap_fraction,
                    "alpha": alpha,
                    "lambda": lambda_value,
                    "selected_h": selected_h,
                    "selected_index": selected_index,
                    "close_pair_h_left": float(h_grid[i]),
                    "close_pair_h_right": float(h_grid[i + 1]),
                    "delta_psi": mesh_delta_psi,
                    "Q_i": Q_i,
                    "close_pair_divergence": close_pair_divergence,
                    "regular_pair_divergence": regular_pair_divergence,
                    "close_pair_similarity": close_pair_similarity,
                    "regular_pair_similarity": regular_pair_similarity,
                    "log_similarity_ratio": log_similarity_ratio,
                    "predicted_log_similarity_ratio": predicted_log_similarity_ratio,
                    "lambda_delta_psi_squared": lambda_delta_psi_squared,
                }
            )
mesh_stress_results = pd.DataFrame(mesh_rows)
mesh_elapsed = time.perf_counter() - mesh_start_time
print(
    f"Completed {len(mesh_stress_results)} mesh-stress selector computations in {mesh_elapsed:.2f} seconds."
)
mesh_baseline = mesh_stress_results[np.isclose(mesh_stress_results["gap_fraction"], 1.0)][
    ["N", "alpha", "selected_h"]
].rename(columns={"selected_h": "baseline_selected_h"})
mesh_stress_results = mesh_stress_results.merge(mesh_baseline, on=["N", "alpha"], how="left")
mesh_stress_results["selector_sensitivity"] = np.abs(
    mesh_stress_results["selected_h"] - mesh_stress_results["baseline_selected_h"]
)
mesh_max_sensitivity = (
    mesh_stress_results.groupby(["N", "alpha"], as_index=False)["selector_sensitivity"]
    .max()
    .rename(columns={"selector_sensitivity": "max_selector_sensitivity"})
)
display(
    mesh_max_sensitivity.pivot(index="alpha", columns="N", values="max_selector_sensitivity").round(
        6
    )
)
DISPLAY_N = 400
display_data = mesh_stress_results[mesh_stress_results["N"] == DISPLAY_N]
fig, ax = plt.subplots(figsize=(13, 8))
for alpha in MESH_ALPHA_VALUES:
    subset = display_data[np.isclose(display_data["alpha"], alpha)].sort_values("gap_fraction")
    ax.plot(
        subset["gap_fraction"],
        subset["selector_sensitivity"],
        marker="o",
        markersize=4,
        linewidth=1.4,
        label=f"$\\alpha={alpha:.1f}$",
        color=MESH_ALPHA_COLORS[float(alpha)],
    )
ax.set_xlabel("compressed gap fraction $c$")
ax.set_ylabel("$|h_{\\rm spec}(c)-h_{\\rm spec}(1)|$")
ax.set_title(f"Mesh sensitivity at $N={DISPLAY_N}$, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(11, 7))
for current_N in sorted(mesh_max_sensitivity["N"].unique()):
    subset = mesh_max_sensitivity[mesh_max_sensitivity["N"] == current_N].sort_values("alpha")
    ax.plot(
        subset["alpha"],
        subset["max_selector_sensitivity"],
        marker="o",
        markersize=4,
        linewidth=1.8,
        label=f"$N={current_N}$",
    )
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel("$\\max_c |h_{\\rm spec}(c)-h_{\\rm spec}(1)|$")
ax.set_title(f"Sensitivity to a microscopic mesh defect, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
STRESS_GAP = 0.2
stress_data = mesh_stress_results[np.isclose(mesh_stress_results["gap_fraction"], STRESS_GAP)]
fig, ax = plt.subplots(figsize=(11, 7))
for current_N in sorted(stress_data["N"].unique()):
    subset = stress_data[stress_data["N"] == current_N].sort_values("alpha")
    ax.plot(
        subset["alpha"],
        subset["log_similarity_ratio"],
        marker="o",
        markersize=4,
        linewidth=1.8,
        label=f"$N={current_N}$",
    )
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.axhline(0.0, linestyle=":", linewidth=1.0, color="black")
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel("$\\log\\left(S_{\\rm close}/S_{\\rm regular}\\right)$")
ax.set_title(f"Similarity advantage of compressed gap $c={STRESS_GAP}$")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
scale_data = mesh_stress_results[np.isclose(mesh_stress_results["gap_fraction"], 1.0)].copy()
fig, ax = plt.subplots(figsize=(11, 7))
for current_N in sorted(scale_data["N"].unique()):
    subset = scale_data[scale_data["N"] == current_N].sort_values("alpha")
    ax.plot(
        subset["alpha"],
        subset["lambda_delta_psi_squared"],
        marker="o",
        markersize=4,
        linewidth=1.8,
        label=f"$N={current_N}$",
    )
ax.axvline(2.0, linestyle="--", linewidth=1.5, color="black", label="critical $\\alpha=2$")
ax.axhline(1.0, linestyle=":", linewidth=1.0, color="black")
ax.set_yscale("log")
ax.set_xlabel("exponent $\\alpha$ in $\\lambda_N=N^\\alpha$")
ax.set_ylabel("$\\lambda_N\\Delta_\\psi^2$")
ax.set_title(f"Critical mesh scale, $\\beta={CUSTOM_BETA:.6f}$")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


### Reading the mesh-robustness comparison

Each colored path corresponds to one of the six named exponents. A horizontal path indicates that the selected bandwidth is insensitive to the isolated close pair. A jump toward $h\approx0.7$ indicates mesh locking.

The theoretical expectation is that exponents below two are increasingly well resolved as $N$ grows, whereas exponents above two become increasingly dependent on individual neighboring grid points. The exact perturbation level at which a path jumps is a finite-$N$ robustness diagnostic, not a new asymptotic boundary.

This experiment remains outside the perfectly uniform-grid assumption. Its purpose is to distinguish a resolved continuous-kernel selection from a fragile nearest-neighbor selection when both happen to return approximately $h_\infty$ on the ideal grid.


## Fixed-sample convergence error

The custom-beta reference paths are computed with the same shared function used in notebook 03.


In [ ]:
custom_beta_results, custom_beta_h_infinity = run_custom_beta_paths(
    beta=CUSTOM_BETA,
    lambda_rules=CUSTOM_BETA_LAMBDA_RULES,
    grid_sizes=CUSTOM_BETA_N_VALUES,
    regime_order=CUSTOM_BETA_REGIME_ORDER,
    search=ENERGY_SEARCH,
    half_width=FFT_HALF_WIDTH,
    spatial_points=FFT_SPATIAL_POINTS,
    h_max=H_MAX,
    h_min=H_MIN,
    samples=samples,
    squared_differences=squared_differences,
)


In [ ]:
fixed_error_results = custom_beta_results.copy()
fixed_error_results["alpha"] = np.log(fixed_error_results["lambda"]) / np.log(
    fixed_error_results["N"]
)
fixed_error_results = (
    fixed_error_results[np.isclose(fixed_error_results["alpha"], 1.0, rtol=0.0, atol=1e-10)]
    .sort_values("N")
    .copy()
)
psi_reference = psi_beta(custom_beta_h_infinity, CUSTOM_BETA)
fixed_error_results["psi_error"] = [
    abs(psi_beta(h, CUSTOM_BETA) - psi_reference) for h in fixed_error_results["selected_h"]
]
psi_interval_length = abs(psi_beta(H_MAX, CUSTOM_BETA) - psi_beta(H_MIN, CUSTOM_BETA))
fixed_error_results["delta_psi"] = psi_interval_length / (fixed_error_results["N"] - 1)
fixed_error_results["error_in_grid_cells"] = (
    fixed_error_results["psi_error"] / fixed_error_results["delta_psi"]
)
fig, ax = plt.subplots(figsize=(9.5, 5.8))
ax.plot(
    fixed_error_results["N"],
    np.maximum(fixed_error_results["psi_error"].to_numpy(), 1e-16),
    marker="o",
    linewidth=2.2,
    label="$|\\psi_\\beta(h_{\\rm Perron})-\\psi_\\beta(\\hat h_Q)|$",
)
ax.plot(
    fixed_error_results["N"],
    fixed_error_results["delta_psi"],
    linestyle="--",
    linewidth=1.8,
    label="grid spacing $\\Delta_\\psi$",
)
ax.set(
    xscale="log",
    yscale="log",
    xlabel="number of bandwidths $N$",
    ylabel="error in $\\psi_\\beta$ coordinate",
    title=f"Fixed-sample Perron convergence to $\\arg\\min \\widehat Q_{{n,\\beta}}$, $\\beta={CUSTOM_BETA:g}$",
)
ax.legend()
fig.tight_layout()
plt.show()
display(
    fixed_error_results[
        [
            "N",
            "lambda",
            "selected_h",
            "continuous_reference",
            "psi_error",
            "delta_psi",
            "error_in_grid_cells",
        ]
    ].round(8)
)


## Local row-sum mechanism


In [ ]:
MECHANISM_N = 1000
MECHANISM_ALPHA = 1.0
MECHANISM_LAMBDA = float(MECHANISM_N**MECHANISM_ALPHA)
mechanism_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=H_MIN, h_max=H_MAX, beta=CUSTOM_BETA, grid_size=MECHANISM_N
)
mechanism_psi_grid = np.array([psi_beta(h, CUSTOM_BETA) for h in mechanism_h_grid])
mechanism_delta_psi = float(mechanism_psi_grid[1] - mechanism_psi_grid[0])
mechanism_divergence = beta2_divergence_matrix_fourier(
    samples, mechanism_h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
)
mechanism_divergence = np.maximum(mechanism_divergence, 0.0)
mechanism_similarity = np.exp(-MECHANISM_LAMBDA * mechanism_divergence)
mechanism_row_integral = mechanism_delta_psi * mechanism_similarity.sum(axis=1)
mechanism_Q = np.array(
    [local_custom_beta_metric(h, squared_differences, beta=CUSTOM_BETA) for h in mechanism_h_grid]
)
mechanism_gaussian_prediction = np.sqrt(np.pi / (MECHANISM_LAMBDA * mechanism_Q))
mechanism_perron_vector = perron_vector_from_divergence(mechanism_divergence, MECHANISM_LAMBDA)
mechanism_h_Q = float(mechanism_h_grid[np.argmin(mechanism_Q)])
mechanism_h_row = float(mechanism_h_grid[np.argmax(mechanism_row_integral)])
mechanism_h_perron = float(mechanism_h_grid[perron_peak_index(mechanism_perron_vector)])
fig, ax = plt.subplots(figsize=(9.5, 5.8))
ax.plot(
    mechanism_h_grid, mechanism_row_integral, linewidth=2.2, label="$\\Delta_\\psi\\sum_j S_{ij}$"
)
ax.plot(
    mechanism_h_grid,
    mechanism_gaussian_prediction,
    linestyle="--",
    linewidth=2.0,
    label="$\\sqrt{\\pi/(\\lambda Q_i)}$",
)
ax.axvline(
    mechanism_h_Q, linestyle=":", linewidth=1.8, label=f"$\\arg\\min Q_i={mechanism_h_Q:.4f}$"
)
ax.axvline(
    mechanism_h_row, linestyle="-.", linewidth=1.6, label=f"$\\arg\\max R_i={mechanism_h_row:.4f}$"
)
ax.axvline(
    mechanism_h_perron,
    linestyle=(0, (5, 2)),
    linewidth=1.6,
    label=f"Perron peak $={mechanism_h_perron:.4f}$",
)
ax.set(
    xscale="log",
    xlabel="bandwidth $h$",
    ylabel="localized row mass",
    title=f"Row-sum mechanism, $N={MECHANISM_N}$, $\\lambda=N^{{{MECHANISM_ALPHA:g}}}$, $\\beta={CUSTOM_BETA:g}$",
)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()
mechanism_summary = pd.DataFrame(
    {
        "quantity": ["argmin local Q", "argmax row sum", "Perron peak"],
        "bandwidth": [mechanism_h_Q, mechanism_h_row, mechanism_h_perron],
    }
)
display(mechanism_summary.round(8))


## Local minima and interval endpoints: discussion control

This fixed-sample control supports the chapter's local-minimum discussion TODO.
It shows detected interior minima and both endpoint comparisons together;
it does not add a second Monte Carlo distribution study.

Sources: [shared search](../kde_tools/energy_minimization.py),
[Gaussian energy](../kde_tools/qhat.py), and [recorded input](../inputs/README.md).


In [ ]:
plt.style.use("default")
KDE_DIR = code_directory
NOTEBOOK_NAME = "continuous_ise_qhat_distributions_vs_n"
pdf_directory = pdf_output_directory


In [ ]:
SEARCH = EnergySearchConfig()
H_MIN, H_MAX = (0.01, 2.0)
R_VALUES = (0.0, 13 / 14, 1.0)
sample_path = REFERENCE_SAMPLE_PATH
samples, _ = load_reference_sample()
objective = GaussianQhatObjective(samples)
analyses = {str(r): minimize_qhat(objective, H_MIN, H_MAX, r=r, config=SEARCH) for r in R_VALUES}
summary = pd.DataFrame(
    [
        dict(
            r=r,
            local_minima=len(analyses[str(r)]["local_minima"]),
            interval_minimum_h=analyses[str(r)]["global_minimum"]["h"],
            interval_minimum_energy=analyses[str(r)]["global_minimum"]["energy"],
            boundary=analyses[str(r)]["global_minimum"]["boundary"],
            resolution_stable=analyses[str(r)]["diagnostics"]["resolution_stable"],
        )
        for r in R_VALUES
    ]
)
display(summary)


In [ ]:
local_minima = pd.DataFrame(
    [dict(r=r, **point) for r in R_VALUES for point in analyses[str(r)]["local_minima"]]
)
display(local_minima)
print("Search settings:", energy_search_settings(SEARCH))
settings = dict(
    search=energy_search_settings(SEARCH),
    h_min=H_MIN,
    h_max=H_MAX,
    r_values=R_VALUES,
    samples_sha256=hashlib.sha256(samples.tobytes()).hexdigest(),
)
run_id = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
output_directory = KDE_DIR / "outputs/data" / NOTEBOOK_NAME / "local_minimum_control" / run_id
output_directory.mkdir(parents=True, exist_ok=True)
(output_directory / "analysis.json").write_text(
    json.dumps(dict(settings=settings, analyses=analyses), indent=2) + "\n"
)
summary.to_csv(output_directory / "summary.csv", index=False)
local_minima.to_csv(output_directory / "local_minima.csv", index=False)


In [ ]:
h = np.geomspace(H_MIN, H_MAX, 601)
fig, axes = plt.subplots(1, len(R_VALUES), figsize=(13, 4.2), constrained_layout=True)
for ax, r in zip(np.atleast_1d(axes), R_VALUES):
    report = analyses[str(r)]
    ax.loglog(h, [objective.energy(x, r) for x in h], color="#24658a")
    for i, point in enumerate(report["local_minima"]):
        ax.scatter(
            point["h"],
            point["energy"],
            color="#d88321",
            s=40,
            label="Interior local minima" if i == 0 else None,
            zorder=3,
        )
    best = report["global_minimum"]
    ax.scatter(
        best["h"],
        best["energy"],
        color="#744da9",
        marker="*",
        s=115,
        label="Interval minimum",
        zorder=4,
    )
    ax.set(xlabel="Bandwidth h", ylabel="Energy", title=f"r = {r:.6g}")
    ax.legend(fontsize=8)
fig.savefig(output_directory / "energy_minima.png", dpi=160)
plt.show()
plt.close(fig)
print("Full reports:", output_directory / "analysis.json")
print("Automatic PDF exports:", pdf_directory)
